# ProductMeasure: Combining Independent True Measures

## Motivation

Sometimes we want a product distribution made from independent components. Each component may already be available as a QMCPy true measure, possibly with its own dimension and transform.

`ProductMeasure` lets us combine these components side by side. If the child dimensions are $d_1, \ldots, d_k$, then the total dimension is

$$
d = d_1 + \cdots + d_k.
$$

A $d$-dimensional QMC point is split into child coordinate blocks, each child transform is applied to its block, and the transformed blocks are concatenated.

In [ ]:

from pathlib import Path
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "qmcpy").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not find the QMCSoftware repository root.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qmcpy.discrete_distribution import DigitalNetB2
from qmcpy.true_measure import Gaussian, ProductMeasure, SciPyWrapper, Uniform, ZeroInflatedExpUniform

warnings.filterwarnings(
    "ignore",
    message="SciPyWrapper joint distribution has no 'logpdf'.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="Custom univariate distribution has no 'pdf' or 'logpdf'.*",
    category=UserWarning,
)

FIG_DIR = REPO_ROOT / "demos" / "product_measure" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=4, suppress=True)


def dummy_child_sampler(d, seed=0):
    return DigitalNetB2(d, seed=seed)


def save_fig(fig, name):
    path = FIG_DIR / name
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path.relative_to(REPO_ROOT)}")


## Important note on child samplers

QMCPy currently requires every true measure to be constructed with a discrete distribution. In this notebook, `dummy_child_sampler` is used only to satisfy that construction requirement.

`ProductMeasure` does **not** use those dummy child samplers to generate product samples. The only sampler that controls product sampling is the `outer_sampler` passed directly to `ProductMeasure`.

The child true measures provide dimension, range, transform, and weight behavior.

## Example 1: Simple product of two 1D true measures

This example combines two one-dimensional true measures. The outer sampler is two-dimensional, so its first coordinate block is sent to child 1 and its second coordinate block is sent to child 2.

In [ ]:

n = 1024
children = [
    Uniform(dummy_child_sampler(1), lower_bound=0.0, upper_bound=2.0),
    Uniform(dummy_child_sampler(1), lower_bound=10.0, upper_bound=12.0),
]
outer_sampler = DigitalNetB2(2, seed=17)
product_measure = ProductMeasure(sampler=outer_sampler, children=children)

x = product_measure(n)
print("sample shape:", x.shape)
print("first coordinate range:", (float(x[:, 0].min()), float(x[:, 0].max())))
print("second coordinate range:", (float(x[:, 1].min()), float(x[:, 1].max())))

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))
axes[0].hist(x[:, 0], bins=30, color="#2563eb", edgecolor="white")
axes[0].set_title("child 1: Uniform(0, 2)")
axes[0].set_xlabel("x0")
axes[1].hist(x[:, 1], bins=30, color="#f97316", edgecolor="white")
axes[1].set_title("child 2: Uniform(10, 12)")
axes[1].set_xlabel("x1")
save_fig(fig, "example1_two_1d_children.png")
plt.show()


The output has shape `(n, 2)`. The two coordinates land in the ranges specified by the two child true measures.

## Example 2: Replications

Replications belong to the outer sampler. Here the first axis indexes replications, the second axis indexes sample number, and the final axis indexes coordinates.

In [ ]:

n = 512
children = [
    Uniform(dummy_child_sampler(1), lower_bound=0.0, upper_bound=2.0),
    Uniform(dummy_child_sampler(1), lower_bound=10.0, upper_bound=12.0),
]
outer_sampler = DigitalNetB2(2, seed=25, replications=3)
product_measure = ProductMeasure(sampler=outer_sampler, children=children)

x = product_measure(n)
print("sample shape:", x.shape)
print("mean by replication:")
print(np.mean(x, axis=1))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(np.arange(1, 4), np.mean(x[..., 0], axis=1), marker="o", label="coord 0 mean")
ax.plot(np.arange(1, 4), np.mean(x[..., 1], axis=1), marker="o", label="coord 1 mean")
ax.set_xticks(np.arange(1, 4))
ax.set_xlabel("replication")
ax.set_ylabel("sample mean")
ax.set_title("Means by replication")
ax.legend()
save_fig(fig, "example2_replication_means.png")
plt.show()


The shape `(3, n, 2)` confirms that `ProductMeasure` preserves replicated sample structure.

## Example 3: Multidimensional child

A child may itself be multidimensional. Here a 2D Gaussian child is combined with a 1D `ZeroInflatedExpUniform` child, giving a 3D product measure. `ZeroInflatedExpUniform` is currently one-dimensional, so it appears as a one-coordinate block.

In [ ]:

n = 2048
p_zero = 0.4
children = [
    Gaussian(
        dummy_child_sampler(2),
        mean=[0.0, 0.0],
        covariance=[[1.0, 0.6], [0.6, 1.0]],
    ),
    ZeroInflatedExpUniform(dummy_child_sampler(1), p_zero=p_zero, lam=1.5),
]
outer_sampler = DigitalNetB2(3, seed=31)
product_measure = ProductMeasure(sampler=outer_sampler, children=children)

x = product_measure(n)
zero_rate = np.mean(x[:, 2] == 0.0)
print("sample shape:", x.shape)
print("Gaussian block shape:", x[:, :2].shape)
print("zero-inflated block shape:", x[:, 2:].shape)
print(f"zero rate in third coordinate: {zero_rate:.3f} (target about {p_zero:.1f})")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(x[:, 0], x[:, 1], s=8, alpha=0.4, color="#2563eb")
axes[0].set_title("2D Gaussian block")
axes[0].set_xlabel("x0")
axes[0].set_ylabel("x1")
axes[1].hist(x[:, 2], bins=40, color="#16a34a", edgecolor="white")
axes[1].set_title("1D zero-inflated block")
axes[1].set_xlabel("x2")
save_fig(fig, "example3_multidimensional_child.png")
plt.show()


The first two columns are generated by the Gaussian child transform. The third column is generated by the zero-inflated child transform.

## Example 4: Verification of which sampler controls samples

This example directly addresses the construction-sampler issue. We build two `ProductMeasure` objects with the same outer sampler seed but different dummy child sampler seeds. The generated samples are identical. Then we change only the outer sampler seed, and the samples change.

This is not saying child samplers are unnecessary in the current QMCPy API. They are still needed to construct the child true measures. It only shows that `ProductMeasure` does not sample from them when generating product samples.

In [ ]:

n = 32

children_a = [
    Uniform(dummy_child_sampler(1, seed=1), lower_bound=0.0, upper_bound=2.0),
    Uniform(dummy_child_sampler(1, seed=2), lower_bound=10.0, upper_bound=12.0),
]
children_b = [
    Uniform(dummy_child_sampler(1, seed=1001), lower_bound=0.0, upper_bound=2.0),
    Uniform(dummy_child_sampler(1, seed=1002), lower_bound=10.0, upper_bound=12.0),
]

same_outer_a = ProductMeasure(sampler=DigitalNetB2(2, seed=101), children=children_a)
same_outer_b = ProductMeasure(sampler=DigitalNetB2(2, seed=101), children=children_b)
different_outer = ProductMeasure(sampler=DigitalNetB2(2, seed=102), children=children_b)

x_a = same_outer_a(n)
x_b = same_outer_b(n)
x_c = different_outer(n)

print("same outer sampler, different child sampler seeds:", np.max(np.abs(x_a - x_b)))
print("different outer sampler seed, same child definitions:", np.max(np.abs(x_b - x_c)))


The first maximum difference is zero. The second is positive because the outer sampler changed.

## Example 5: SciPyWrapper comparison

For simple independent one-dimensional SciPy marginals, one `SciPyWrapper` with a list of marginals is often enough. `ProductMeasure` can match that construction when both use the same total-dimensional outer sampler, but it is more general because children may themselves be multidimensional true measures.

In [ ]:

n = 4096
marginals = [stats.norm(loc=1.0, scale=2.0), stats.gamma(a=4.0, scale=0.5)]
seed = 55

children = [
    SciPyWrapper(dummy_child_sampler(1), marginals[0]),
    SciPyWrapper(dummy_child_sampler(1), marginals[1]),
]
product_measure = ProductMeasure(sampler=DigitalNetB2(2, seed=seed), children=children)
single_wrapper = SciPyWrapper(DigitalNetB2(2, seed=seed), marginals)

x_product = product_measure(n)
x_wrapper = single_wrapper(n)

print("ProductMeasure shape:", x_product.shape)
print("single SciPyWrapper shape:", x_wrapper.shape)
print("ProductMeasure means:", np.mean(x_product, axis=0))
print("single SciPyWrapper means:", np.mean(x_wrapper, axis=0))
print(f"ProductMeasure correlation: {np.corrcoef(x_product.T)[0, 1]:.4f}")
print(f"single SciPyWrapper correlation: {np.corrcoef(x_wrapper.T)[0, 1]:.4f}")
print(f"maximum absolute sample difference: {np.max(np.abs(x_product - x_wrapper)):.3e}")

fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.scatter(x_product[:, 0], x_product[:, 1], s=8, alpha=0.35, label="ProductMeasure")
ax.scatter(x_wrapper[:, 0] + 0.02, x_wrapper[:, 1], s=8, alpha=0.25, label="single SciPyWrapper")
ax.set_xlabel("normal marginal")
ax.set_ylabel("gamma marginal")
ax.set_title("Same sampler, equivalent simple marginals")
ax.legend(markerscale=2)
save_fig(fig, "example5_product_vs_single_scipywrapper.png")
plt.show()


The two constructions agree for this simple case. The scatter plot shifts the `SciPyWrapper` points slightly in the first coordinate so both layers are visible.

## Example 6: Error handling

`ProductMeasure` expects one total-dimensional sampler and a nonempty list or tuple of true measure children. Invalid inputs raise errors before sampling starts.

In [ ]:

bad_cases = [
    ("empty child list", DigitalNetB2(1, seed=7), []),
    ("non-true-measure child", DigitalNetB2(1, seed=7), [object()]),
    ("dimension mismatch", DigitalNetB2(2, seed=7), [Uniform(dummy_child_sampler(1))]),
]
for label, sampler, children in bad_cases:
    try:
        ProductMeasure(sampler=sampler, children=children)
    except Exception as err:
        print(f"{label}: {type(err).__name__}: {err}")


These checks catch common setup mistakes: forgetting children, passing objects that are not true measures, or using a sampler whose dimension does not match the sum of child dimensions.

More general chained transformations for importance sampling are a separate design issue and are not addressed by `ProductMeasure`.